# Konwersja z json na csv

In [21]:
# import json
# import csv

# # 1. Wczytaj plik JSON wyeksportowany z Tiled
# with open('/home/tomasz/GitProjekty/Mapa2/Mapa2.json', 'r') as f:
#     tiled_data = json.load(f)

# # 2. Pobierz parametry mapy
# width = tiled_data['width']         # Szerokość mapy w kafelkach
# height = tiled_data['height']       # Wysokość mapy w kafelkach (potrzebna do odwrócenia Y)
# tile_width = tiled_data['tilewidth']   # Szerokość kafelka w px
# tile_height = tiled_data['tileheight'] # Wysokość kafelka w px
# layer_data = tiled_data['layers'][0]['data'] 

# csv_rows = []

# # 3. Przeiteruj po całej płaskiej tablicy kafelków
# for index, tile_id in enumerate(layer_data):
#     if tile_id > 0:  # Pomijamy puste pola (0)
        
#         # Standardowe współrzędne Tiled (0,0 w lewym górnym rogu)
#         grid_x = index % width
#         grid_y_top_down = index // width
        
#         # ODWRÓCENIE OSI Y: Teraz 0 będzie na samym dole mapy, a najwyższy wiersz to height-1
#         grid_y_bottom_up = (height - 1) - grid_y_top_down
        
#         # OPCJONALNIE (w pikselach od dołu):
#         # pixel_x = grid_x * tile_width
#         # pixel_y_bottom_up = grid_y_bottom_up * tile_height
        
#         # Zapisujemy do CSV z odwróconym Y
#         csv_rows.append([tile_id, grid_x, grid_y_bottom_up])

# # 4. Zapis do pliku CSV
# with open('/home/tomasz/GitProjekty/Mapa2/mapa_kafelkow.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(['ID', 'X', 'Y']) 
#     writer.writerows(csv_rows)

# print(f"Wygenerowano strukturę dla {len(csv_rows)} kafelków (Y rośnie do góry).")



import json

# 1. Wczytaj plik JSON wyeksportowany z Tiled
with open('/home/tomasz/GitProjekty/Mapa2/Mapa2.json', 'r') as f:
    tiled_data = json.load(f)

# 2. Parametry mapy
width = tiled_data['width']
height = tiled_data['height']
tile_width = tiled_data['tilewidth']   # Szerokość kafelka w px
tile_height = tiled_data['tileheight'] # Wysokość kafelka w px
layer_data = tiled_data['layers'][0]['data']

c_lines = []

# 3. Przeiteruj po kafelkach i zbuduj linie kodu C
for index, tile_id in enumerate(layer_data):
    if tile_id > 0:  # Pomijamy puste pola (0)
        grid_x = index % width
        grid_y_top_down = index // width
        
        # Odwrócenie osi Y (dół to 0, góra to height-1)
        grid_y_bottom_up = (height - 1) - grid_y_top_down
        
        # Tworzymy formatowanie dokładnie pod strukturę w C
        line = f"    {{ .spriteID = {tile_id}, .x = {grid_x * tile_width}, .y = {grid_y_bottom_up * tile_height} }}"
        c_lines.append(line)

# Połączenie wszystkich elementów przecinkami i nowymi liniami
# Ostatni element też może mieć przecinek w C (jest to w pełni poprawne)
c_array_content = ",\n".join(c_lines)

# 4. Generowanie pełnego bloku kodu do wklejenia
kod_c = f"""// Liczba kafelków: {len(c_lines)}
Tile mapa[{len(c_lines)}] = {{
{c_array_content}
}};"""

# 5. Zapisz do pliku (na wypadek gdyby mapa była ogromna)
with open('/home/tomasz/GitProjekty/Mapa2/mapa_kafelkow.txt', 'w') as f:
    f.write(kod_c)

# 6. Wypisz wynik w Jupyterze, żeby można było od razu skopiować
print(kod_c)

// Liczba kafelków: 8
Tile mapa[8] = {
    { .spriteID = 2, .x = 96, .y = 208 },
    { .spriteID = 3, .x = 192, .y = 208 },
    { .spriteID = 1, .x = 336, .y = 208 },
    { .spriteID = 2, .x = 304, .y = 144 },
    { .spriteID = 3, .x = 96, .y = 128 },
    { .spriteID = 3, .x = 176, .y = 96 },
    { .spriteID = 1, .x = 272, .y = 80 },
    { .spriteID = 1, .x = 64, .y = 32 }
};
